In [ ]:
CHECKPOINT = "final_results.json"
all_results = {}
if os.path.exists(CHECKPOINT):
    try:
        with open(CHECKPOINT) as f: all_results = json.load(f)
        print(f"Resumed: {len(all_results)} models from checkpoint")
    except: all_results = {}

device = torch.device("cuda")

for fams, big in [(MODELS_7B, True), (MODELS_SMALL, False)]:
    for bid, iid, name in fams:
        for mid, mn in [(bid, name)] + ([(iid, f"{name}-IT")] if iid else []):
            if mn in all_results and all(v is not None for v in all_results[mn].values()):
                print(f"Skipping {mn} (done)"); continue
            print(f"\n{'='*60}\n{mn}  ({mid})\n{'='*60}")
            model = None
            try:
                kw = {"quantization_config": BNB_CONFIG, "device_map": "auto"} if (big and USE_4BIT) else \
                     {"torch_dtype": torch.float16, "device_map": "auto", "max_memory": {0:"14.5GiB", "cpu":"32GiB"}, "offload_folder": "off"} if big else \
                     {"torch_dtype": torch.float16, "device_map": "auto"}
                tok = AutoTokenizer.from_pretrained(mid, token=os.environ["HF_TOKEN"])
                if tok.pad_token is None: tok.pad_token = tok.eos_token
                model = AutoModelForCausalLM.from_pretrained(mid, **kw, token=os.environ["HF_TOKEN"])
                model.eval()
                print(f"Loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")
            except Exception as e:
                print(f"FAIL: {e}"); continue
            ct = mid in DEEPSEEK_IDS; sc = {pt:{} for pt in PROBE_VARIANTS}
            for pt, vv in PROBE_VARIANTS.items():
                for pv in vv:
                    its = []
                    for idx, it in enumerate(ITEMS):
                        p = make_prompt(it, pt, pv)
                        if ct:
                            p = tok.apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
                        reps = []
                        for _ in range(3):
                            try:
                                s = score_model(model, tok, p, pv, device)
                                if s and 1<=s<=5: reps.append(s)
                            except: pass
                        if reps: its.append(np.mean(reps))
                        if (idx+1)%25==0: print(f"  [{pt}/{pv}] {idx+1}/{len(ITEMS)} ({len(its)} scored)", flush=True)
                    sc[pt][pv] = {"mean": float(np.mean(its)) if its else None}
                    if not its:
                        p2 = make_prompt(ITEMS[0], pt, pv)
                        if ct:
                            p2 = tok.apply_chat_template([{"role":"user","content":p2}], tokenize=False, add_generation_prompt=True)
                        inp = tok(p2, return_tensors="pt", truncation=True, max_length=1024).to(device)
                        with torch.no_grad():
                            o = model.generate(**inp, max_new_tokens=25, temperature=0.0, do_sample=False, pad_token_id=tok.eos_token_id)
                        d = tok.decode(o[0], skip_special_tokens=True)
                        print(f"  ZERO: \"{d[len(p2):].strip()[:150]}\")
            r = {}
            for pt, vv in PROBE_VARIANTS.items():
                ms = [sc[pt][v]["mean"] for v in vv if sc[pt][v]["mean"] is not None]
                r[pt] = round(max(ms)-min(ms),2) if len(ms)>=2 else None
            all_results[mn] = r
            print(f"DONE: {r}")
            with open(CHECKPOINT, "w") as f: json.dump(all_results, f, indent=2)
            del model; gc.collect(); torch.cuda.empty_cache(); time.sleep(3)

with open(CHECKPOINT, "w") as f: json.dump(all_results, f, indent=2)
print(f"\nDONE: {len(all_results)} models")
